In [ ]:
#Installations needed to evaluate the models
#!pip install evaluate
#!pip install git+https://github.com/google-research/bleurt.git
#!pip install rouge_score

In [ ]:
#Import the TL;DR dataset.
from datasets import load_dataset
posts = load_dataset("trl-lib/tldr")

c:\Users\Tobias\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Method for getting the subreddit from the prompt text.
def getSubredditFromPost(post):
    return  post['prompt'].split('\n')[0][11:];

#Get all subreddits and the amount of posts present within each subreddit.
counts = {}
subreddits = []
for post in posts['train']:
    subreddit = getSubredditFromPost(post)
    if subreddit not in subreddits:
        subreddits.append(subreddit)
    if(subreddit in counts):
        counts[subreddit] = counts[subreddit] + 1
    else:
        counts[subreddit] = 1

#To ensure equal representation of subreddits we evaluate the same amount of posts from each subreddit. 
#That means we will take the amount of posts in the subreddit that has fewest posts and evaluate this 
#amount of posts for each subreddit.
amount_of_posts_to_evaluate = min(counts.values())


In [ ]:
# This is the NOIR metric which has been retreived from the repository https://github.com/afoland/NOIR
# (C) Andrew Foland, Sonnetiq, 2024; license granted under Apache 2.0
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(vector1, vector2):
    # Calculate cosine similarity between two vectors
    return cosine_similarity([vector1], [vector2])[0][0]

def embed_string(text, model, tokenizer):
    input_ids = tokenizer.encode(text, return_tensors='pt', max_length=512, truncation=True)
    with torch.no_grad():
        embedding = model(input_ids).last_hidden_state.mean(dim=1).squeeze().tolist()
    return embedding

noir_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
noir_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
def NOIR(text, summary):
    text_embedding = embed_string(text, noir_model, noir_tokenizer)
    summary_embedding = embed_string(summary, noir_model, noir_tokenizer)
    D = calculate_cosine_similarity(text_embedding, summary_embedding)

    text_length = len(noir_tokenizer.encode(text))
    summary_length = len(noir_tokenizer.encode(summary))
    k = summary_length / text_length

    sque_metric = np.log(k) / np.log(D)
    return sque_metric

In [ ]:
import time
import evaluate

#Load the rouge and bleurt metric for evaluation.
rouge = evaluate.load("rouge")
bleurt = evaluate.load("bleurt", module_type="metric")

#Compute the NOIR, bleurt and rouge scores for the given pipeline.
def compute_scores(pipe):
  noirScores = {}
  bleurtScores = {}
  rougeScores = {}

  for subreddit in subreddits:
    start = time.time()

    noirScores[subreddit] = []

    #Get the prompts and the labels of the given subreddit.
    sub_dataset = posts.filter(lambda post : getSubredditFromPost(post) == subreddit)['train'].select(range(amount_of_posts_to_evaluate))
    sub_dataset_prompts = [p['prompt'] for p in sub_dataset]
    labels = [p['completion'] for p in sub_dataset]
    
    #Generate the summaries from the given pipeline.
    generatedTexts = [text['summary_text'] for text in pipe(sub_dataset_prompts)]

    #Compute the NOIR score for each summary
    for prompt, generatedText in zip(sub_dataset_prompts, generatedTexts):
      noirScore = NOIR(prompt, generatedText)
      noirScores[subreddit].append(noirScore)
    
    #Compute the BLEURT score for each summary
    bleurtScore = bleurt.compute(predictions=generatedTexts, references=labels)
    bleurtScores[subreddit] = bleurtScore['scores']

    #Compute the average ROUGE-1, ROUGE-2, ROUGE-L and ROUGE-Lsum for the generated summaries.
    rougeScore = rouge.compute(predictions=generatedTexts, references=labels)
    rougeScores[subreddit] = rougeScore

    print(subreddit + " time: " + str(time.time()-start))

  return noirScores, bleurtScores, rougeScores


Using default BLEURT-Base checkpoint for sequence maximum length 128. You can use a bigger model for better results with e.g.: evaluate.load('bleurt', 'bleurt-large-512').



INFO:tensorflow:Reading checkpoint C:\Users\Tobias\.cache\huggingface\metrics\bleurt\default\downloads\extracted\30ddb91ee5d94d9d15afa72de293cfbd181c7d7b153f001020ed5092a3320c9d\bleurt-base-128.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint bert_custom
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:bert_custom
INFO:tensorflow:... vocab_file:vocab.txt
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... do_lower_case:True
INFO:tensorflow:... max_seq_length:128
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating WordPiece tokenizer.

INFO:tensorflow:WordPiece tokenizer instantiated.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.
INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [ ]:
#Evaluate the bart base model from ainize
from transformers import pipeline
ainize_noirScores, ainize_bleurtScores, ainize_rougeScores = compute_scores(pipeline("summarization", model="ainize/bart-base-cnn", device=0))



Device set to use cuda:0


Got here 1
Got here 2
Got here 3


Token indices sequence length is longer than the specified maximum sequence length for this model (521 > 512). Running this sequence through the model will result in indexing errors


Got here 4
Got here 5
r/relationships time: 106.48798632621765
Got here 1
Got here 2


Your max_length is set to 128, but your input_length is only 73. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=36)
Your max_length is set to 128, but your input_length is only 121. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=60)


...

Got here 3
Got here 4
Got here 5
r/GetMotivated time: 93.48796010017395
Got here 1
Got here 2


Your max_length is set to 128, but your input_length is only 86. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=43)


Got here 3
Got here 4
Got here 5
r/Parenting time: 106.32328414916992
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/weddingplanning time: 101.30360102653503
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/college time: 95.98733520507812
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/jobs time: 96.4089605808258
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/Dogtraining time: 109.10490226745605
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/Pets time: 105.84723234176636
Got here 1
Got here 2


Your max_length is set to 128, but your input_length is only 107. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=53)


Got here 3
Got here 4
Got here 5
r/Cooking time: 95.09589195251465
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/askwomenadvice time: 107.1855251789093
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/AskDocs time: 105.42297148704529
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/travel time: 92.56241965293884
Got here 1
Got here 2
Got here 3
Got here 4
Got here 5
r/books time: 90.6350314617157


In [ ]:
#Save the scores to a csv and a json file.
import pandas as pd
from transformers import pipeline
df = pd.DataFrame(ainize_noirScores)
df.to_json('ainize_noirScores.json', index=False)
df.to_csv('ainize_noirScores.csv', index=False)
df = pd.DataFrame(ainize_bleurtScores)
df.to_json('ainize_bleurtScores.json', index=False)
df.to_csv('ainize_bleurtScores.csv', index=False)
df = pd.DataFrame(ainize_rougeScores)
df.to_json('ainize_rougeScores.json', index=False)
df.to_csv('ainize_rougeScores.csv', index=False)

{'r/relationships': [-1.2615772485733032, -1.0405648946762085, -1.1745456457138062, -0.8102166056632996, -0.6860062479972839, -1.0360978841781616, -1.2219822406768799, -1.4258077144622803, -0.7331310510635376, -0.444368451833725, -0.9615211486816406, -0.6469079852104187, -0.8974311947822571, -0.8366178870201111, -1.048610806465149, -0.9884893894195557, -0.8084520697593689, -1.2072064876556396, -0.9660484194755554, -1.3985713720321655, -0.8382546901702881, -1.0878865718841553, -0.8062964081764221, -1.3638012409210205, -1.2141013145446777, -0.7908242344856262, -0.6498783230781555, -0.9985195994377136, -1.1246123313903809, -1.0566532611846924, -1.2598469257354736, -0.7185646295547485, -1.1544424295425415, -1.002940058708191, -1.1102917194366455, -0.6827796101570129, -0.6938673853874207, -0.9241867661476135, -0.9650933146476746, -0.7682896256446838, -1.1617246866226196, -0.9089774489402771, -1.3374569416046143, -0.9806615710258484, -0.9045617580413818, -1.141010046005249, -0.91113096475601

In [ ]:
#Evaluate the t5-small model from falconsai.
from transformers import pipeline
falconsai_noirScores, falconsai_bleurtScores, falconsai_rougeScores = compute_scores(pipeline("summarization", model="Falconsai/text_summarization", device=0))


Device set to use cuda:0


Got here 1
Got here 2


Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors
Your max_length is set to 200, but your input_length is only 196. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=98)


Got here 3
Got here 4
Got here 5
r/relationships time: 138.3977518081665
Got here 1
Got here 2


Your max_length is set to 200, but your input_length is only 177. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=88)
Your max_length is set to 200, but your input_length is only 78. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=39)
Your max_length is set to 200, but your input_length is only 199. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Your max_length is set to 200, but your input_length is only 167. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)
Y

...

Got here 3
Got here 4
Got here 5
r/AskDocs time: 153.87722969055176
Got here 1
Got here 2


Your max_length is set to 200, but your input_length is only 189. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=94)
Your max_length is set to 200, but your input_length is only 179. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=89)
Your max_length is set to 200, but your input_length is only 198. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Your max_length is set to 200, but your input_length is only 195. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)


Got here 3
Got here 4
Got here 5
r/travel time: 133.26980876922607
Got here 1
Got here 2


Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 171. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=85)


Got here 3
Got here 4
Got here 5
r/books time: 138.84633016586304


In [ ]:
#Save the scores to a csv and a json file.
import pandas as pd
from transformers import pipeline
df = pd.DataFrame(falconsai_noirScores)
df.to_json('falconsai_noirScores.json', index=False)
df.to_csv('falconsai_noirScores.csv', index=False)
df = pd.DataFrame(falconsai_bleurtScores)
df.to_json('falconsai_bleurtScores.json', index=False)
df.to_csv('falconsai_bleurtScores.csv', index=False)
df = pd.DataFrame(falconsai_rougeScores)
df.to_json('falconsai_rougeScores.json', index=False)
df.to_csv('falconsai_rougeScores.csv', index=False)

In [ ]:
#Evaluate the pegasus large model from google
from transformers import pipeline
pipe = pipeline("summarization", model="google/pegasus-large", device=0)
pegasus_noirScores, pegasus_bleurtScores, pegases_rougeScores = compute_scores(pipe)


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Got here 1
Got here 2


Your max_length is set to 256, but your input_length is only 186. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=93)
Your max_length is set to 256, but your input_length is only 220. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=110)
Your max_length is set to 256, but your input_length is only 177. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=88)
Your max_length is set to 256, but your input_length is only 245. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=122

Got here 3


Token indices sequence length is longer than the specified maximum sequence length for this model (521 > 512). Running this sequence through the model will result in indexing errors


...

Got here 2


Your max_length is set to 256, but your input_length is only 239. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=119)
Your max_length is set to 256, but your input_length is only 186. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=93)
Your max_length is set to 256, but your input_length is only 215. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=107)
Your max_length is set to 256, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97

Got here 3
Got here 4
Got here 5
r/travel time: 255.62330746650696
Got here 1
Got here 2


Your max_length is set to 256, but your input_length is only 218. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=109)
Your max_length is set to 256, but your input_length is only 204. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=102)
Your max_length is set to 256, but your input_length is only 224. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=112)
Your max_length is set to 256, but your input_length is only 193. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9

Got here 3
Got here 4
Got here 5
r/books time: 252.28561115264893


In [ ]:
#Save the scores to a csv and a json file.
import pandas as pd
df = pd.DataFrame(pegasus_noirScores)
df.to_json('pegasus_noirScores.json', index=False)
df.to_csv('pegasus_noirScores.csv', index=False)
df = pd.DataFrame(pegasus_bleurtScores)
df.to_json('pegasus_bleurtScores.json', index=False)
df.to_csv('pegasus_bleurtScores.csv', index=False)
df = pd.DataFrame(pegases_rougeScores)
df.to_json('pegases_rougeScores.json', index=False)
df.to_csv('pegases_rougeScores.csv', index=False)